# PAML Final Project Data Pre-processing

## Import Libraries
### Import
some libs are used for geo-processing or downloading TLC raw data.

In [2]:
import requests
import os
import pandas as pd
import numpy as np
import pyarrow
import geopandas as gpd
from pathlib import Path
from shapely import wkt

## Preprocess TLC Data
### Download TLC data
TLC taxi data is a large dataset (~33.6 GB for 6 years). Therefore, if you need to process TLC raw data, you may need to download data to a local PC rather than Google Drive. If you don't need to process TLC data, just skip this cell.

In [3]:
# --- Config ---
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
TYPES = ["yellow", "green", "fhv", "fhvhv"]

START_YEAR = 2020
END_YEAR = 2025

DOWNLOAD_DIR = Path("data/tlc_data")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# --- Generate all year-month combinations ---
year_months = []
for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        ym = f"{year}-{month:02d}"
        year_months.append(ym)

# --- Download loop ---
failed = []

for t in TYPES:
    type_dir = DOWNLOAD_DIR / t
    type_dir.mkdir(exist_ok=True)

    for ym in year_months:
        filename = f"{t}_tripdata_{ym}.parquet"
        url = f"{BASE_URL}/{filename}"
        save_path = type_dir / filename

        if save_path.exists():
            print(f"[SKIP] {filename}")
            continue

        try:
            print(f"[DOWNLOADING] {filename}")
            r = requests.get(url, stream=True, timeout=30)

            if r.status_code == 200:
                with open(save_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
            else:
                print(f"[FAILED] {filename} (status {r.status_code})")
                failed.append(filename)

        except Exception as e:
            print(f"[ERROR] {filename}: {e}")
            failed.append(filename)

print("\nDownload complete.")
print(f"Failed files: {len(failed)}")
print(failed[:10])

[SKIP] yellow_tripdata_2020-01.parquet
[SKIP] yellow_tripdata_2020-02.parquet
[SKIP] yellow_tripdata_2020-03.parquet
[SKIP] yellow_tripdata_2020-04.parquet
[SKIP] yellow_tripdata_2020-05.parquet
[SKIP] yellow_tripdata_2020-06.parquet
[SKIP] yellow_tripdata_2020-07.parquet
[SKIP] yellow_tripdata_2020-08.parquet
[SKIP] yellow_tripdata_2020-09.parquet
[SKIP] yellow_tripdata_2020-10.parquet
[SKIP] yellow_tripdata_2020-11.parquet
[SKIP] yellow_tripdata_2020-12.parquet
[SKIP] yellow_tripdata_2021-01.parquet
[SKIP] yellow_tripdata_2021-02.parquet
[SKIP] yellow_tripdata_2021-03.parquet
[SKIP] yellow_tripdata_2021-04.parquet
[SKIP] yellow_tripdata_2021-05.parquet
[SKIP] yellow_tripdata_2021-06.parquet
[SKIP] yellow_tripdata_2021-07.parquet
[SKIP] yellow_tripdata_2021-08.parquet
[SKIP] yellow_tripdata_2021-09.parquet
[SKIP] yellow_tripdata_2021-10.parquet
[SKIP] yellow_tripdata_2021-11.parquet
[SKIP] yellow_tripdata_2021-12.parquet
[SKIP] yellow_tripdata_2022-01.parquet
[SKIP] yellow_tripdata_20

### Load TLC taxi data
Read raw data from .parquet files. We combine all 4 traffic data together, meaning that there is no difference between yellow taxi, green taxi and fhv. This cell also counts taxi count in each taxi zone & each day.

In [4]:
PROJECT_ROOT = Path.cwd()
TLC_DIR = PROJECT_ROOT / "data" / "tlc_data"

TYPES = ["yellow", "green", "fhv", "fhvhv"]

START_YEAR = 2020
END_YEAR = 2025

use_cols_map = {
    "yellow": ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID"],
    "green":  ["lpep_pickup_datetime", "lpep_dropoff_datetime", "PULocationID", "DOLocationID"],
    "fhv":    ["pickup_datetime", "dropOff_datetime", "PUlocationID", "DOlocationID"],
    "fhvhv":  ["pickup_datetime", "dropoff_datetime", "PULocationID", "DOLocationID"],
}

all_monthly_results = []

for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        ym = f"{year}-{month:02d}"
        monthly_events = []

        print(f"\nProcessing {ym}...")

        for t in TYPES:
            file_path = TLC_DIR / t / f"{t}_tripdata_{ym}.parquet"

            if not file_path.exists():
                print(f"  [MISSING] {file_path.name}")
                continue

            try:
                print(f"  Loading {t}...")

                cols = use_cols_map[t]
                df = pd.read_parquet(file_path, columns=cols)

                # Standardize column names
                df = df.rename(columns={
                    cols[0]: "pickup_datetime",
                    cols[1]: "dropoff_datetime",
                    cols[2]: "PULocationID",
                    cols[3]: "DOLocationID"
                })

                # Build pickup events
                pickup = df[["pickup_datetime", "PULocationID"]].copy()
                pickup.columns = ["datetime", "LocationID"]

                # Build dropoff events
                dropoff = df[["dropoff_datetime", "DOLocationID"]].copy()
                dropoff.columns = ["datetime", "LocationID"]

                events = pd.concat([pickup, dropoff], ignore_index=True)
                monthly_events.append(events)

            except Exception as e:
                print(f"  [FAILED] {t} {ym}: {e}")
                continue

        if not monthly_events:
            continue

        events = pd.concat(monthly_events, ignore_index=True)

        # Clean
        events["datetime"] = pd.to_datetime(events["datetime"], errors="coerce")
        events["LocationID"] = pd.to_numeric(events["LocationID"], errors="coerce")

        events = events.dropna(subset=["datetime", "LocationID"]).copy()
        events["LocationID"] = events["LocationID"].astype("int32")

        # Date + peak time
        events["date"] = events["datetime"].dt.normalize()
        events["hour"] = events["datetime"].dt.hour
        events["minute"] = events["datetime"].dt.minute

        # Peak definition:
        # morning peak: 07:00–09:30
        # evening peak: 16:00–19:00
        events["is_peak"] = (
            ((events["hour"] == 7) | (events["hour"] == 8) | ((events["hour"] == 9) & (events["minute"] <= 30))) |
            ((events["hour"] == 16) | (events["hour"] == 17) | (events["hour"] == 18) | ((events["hour"] == 19) & (events["minute"] == 0)))
        ).astype("int8")

        # Aggregate to date × taxi_zone_id × is_peak
        monthly_base = (
            events
            .groupby(["date", "LocationID", "is_peak"], as_index=False)
            .size()
            .rename(columns={
                "LocationID": "taxi_zone_id",
                "size": "traffic_count"
            })
        )

        all_monthly_results.append(monthly_base)
        print(f"  Done {ym}, shape={monthly_base.shape}")

# Combine all months
taxi_base = pd.concat(all_monthly_results, ignore_index=True)

# Final aggregation
taxi_base = (
    taxi_base
    .groupby(["date", "taxi_zone_id", "is_peak"], as_index=False)["traffic_count"]
    .sum()
)

print("\n=== Final taxi_base ===")
print(taxi_base.shape)
display(taxi_base.head())


Processing 2020-01...
  Loading yellow...
  Loading green...
  Loading fhv...
  Loading fhvhv...
  Done 2020-01, shape=(16668, 4)

Processing 2020-02...
  Loading yellow...
  Loading green...
  Loading fhv...
  Loading fhvhv...
  Done 2020-02, shape=(15624, 4)

Processing 2020-03...
  Loading yellow...
  Loading green...
  Loading fhv...
  Loading fhvhv...
  Done 2020-03, shape=(16660, 4)

Processing 2020-04...
  Loading yellow...
  Loading green...
  Loading fhv...
  Loading fhvhv...
  Done 2020-04, shape=(15985, 4)

Processing 2020-05...
  Loading yellow...
  Loading green...
  Loading fhv...
  Loading fhvhv...
  Done 2020-05, shape=(16410, 4)

Processing 2020-06...
  Loading yellow...
  Loading green...
  Loading fhv...
  Loading fhvhv...
  Done 2020-06, shape=(15890, 4)

Processing 2020-07...
  Loading yellow...
  Loading green...
  Loading fhv...
  Loading fhvhv...
  Done 2020-07, shape=(16470, 4)

Processing 2020-08...
  Loading yellow...
  Loading green...
  Loading fhv...
  Lo

,date,taxi_zone_id,is_peak,traffic_count
0,1900-01-01,3,0,1
1,1917-09-10,235,0,1
2,1970-01-20,244,0,1
3,2001-01-01,43,0,1
4,2001-01-01,48,0,1


### Convert TLC Taxi Zone to NYC Zipcode
The following cells transferred TLC taxi zones to nyc zipcodes.

In [5]:
PROJECT_ROOT = Path.cwd()

zone_path = PROJECT_ROOT / "data" / "nyc_taxi_zones" / "nyc_taxi_zones.shp"
zip_path  = PROJECT_ROOT / "data" / "nyc_zip" / "nyc_zip.shp"

gdf_zone = gpd.read_file(zone_path)
gdf_zip  = gpd.read_file(zip_path)

# --- Use MODZCTA instead of ZCTA ---
zip_col = "modzcta"
zone_col = "locationid"

# --- Keep only needed columns ---
gdf_zone = gdf_zone[[zone_col, "geometry"]].copy()
gdf_zone = gdf_zone.rename(columns={zone_col: "taxi_zone_id"})

gdf_zip  = gdf_zip[[zip_col, "geometry"]].copy()
gdf_zip  = gdf_zip.rename(columns={zip_col: "zip_code"})

# --- Clean types ---
gdf_zone["taxi_zone_id"] = pd.to_numeric(gdf_zone["taxi_zone_id"], errors="coerce")
gdf_zone = gdf_zone.dropna(subset=["taxi_zone_id"]).copy()
gdf_zone["taxi_zone_id"] = gdf_zone["taxi_zone_id"].astype("int32")

gdf_zip["zip_code"] = (
    gdf_zip["zip_code"]
    .astype(str)
    .str.strip()
)

# --- CRS ---
gdf_zone = gdf_zone.to_crs(epsg=2263)
gdf_zip  = gdf_zip.to_crs(epsg=2263)

# --- Compute zone area ---
gdf_zone["zone_area"] = gdf_zone.geometry.area

# --- Overlay ---
zone_zip_intersection = gpd.overlay(gdf_zone, gdf_zip, how="intersection")

# --- Intersection area ---
zone_zip_intersection["intersect_area"] = zone_zip_intersection.geometry.area

# --- Ensure zone_area exists ---
if "zone_area" not in zone_zip_intersection.columns:
    zone_zip_intersection = zone_zip_intersection.merge(
        gdf_zone[["taxi_zone_id", "zone_area"]],
        on="taxi_zone_id",
        how="left"
    )

# --- Weight ---
zone_zip_intersection["zone_to_zip_weight"] = (
    zone_zip_intersection["intersect_area"] / zone_zip_intersection["zone_area"]
)

# --- Keep columns ---
zone_zip_intersection = zone_zip_intersection[
    ["taxi_zone_id", "zip_code", "intersect_area", "zone_area", "zone_to_zip_weight", "geometry"]
].copy()

print("Zone-ZIP (MODZCTA) overlap created.")
print(zone_zip_intersection.shape)
display(zone_zip_intersection.head())

Zone-ZIP (MODZCTA) overlap created.
(1074, 6)


,taxi_zone_id,zip_code,intersect_area,zone_area,zone_to_zip_weight,geometry
0,2,11693,9.690224e+02,1.439095e+08,6.733552e-06,"MULTIPOLYGON (((1035187.96 163371.88, 1035187...."
1,2,11434,7.364716e+00,1.439095e+08,5.117601e-08,"MULTIPOLYGON (((1042311.166 170442.071, 104231..."
2,2,99999,3.209514e+07,1.439095e+08,2.230230e-01,"MULTIPOLYGON (((1033439.643 170883.946, 103347..."
3,3,10461,1.055927e+03,3.168508e+07,3.332569e-05,"MULTIPOLYGON (((1029273.432 251677.77, 1029112..."
4,3,10467,3.115670e+01,3.168508e+07,9.833238e-07,"MULTIPOLYGON (((1022545.939 254871.041, 102254..."


### Method 1: Max-overlap

In [7]:
# --- 1. Build max-overlap crosswalk ---
max_overlap_map = (
    zone_zip_intersection
    .sort_values(["taxi_zone_id", "intersect_area"], ascending=[True, False])
    .drop_duplicates(subset=["taxi_zone_id"])
    [["taxi_zone_id", "zip_code"]]
    .copy()
)

print("Max-overlap crosswalk:")
display(max_overlap_map.head())

# --- 2. Merge taxi counts to ZIP ---
taxi_zip_max = taxi_base.merge(
    max_overlap_map,
    on="taxi_zone_id",
    how="left"
)

# Drop zones that could not be mapped
taxi_zip_max = taxi_zip_max.dropna(subset=["zip_code"]).copy()
taxi_zip_max["zip_code"] = taxi_zip_max["zip_code"].astype(str)

# --- 3. Aggregate to ZIP × date × is_peak ---
taxi_zip_max = (
    taxi_zip_max
    .groupby(["date", "zip_code", "is_peak"], as_index=False)["traffic_count"]
    .sum()
)

# --- 4. Fill missing ZIP × date × is_peak with 0 ---
taxi_zip_max["date"] = pd.to_datetime(taxi_zip_max["date"], errors="coerce")

start_date = pd.Timestamp("2020-01-01")
end_date   = pd.Timestamp("2025-12-31")

full_dates = pd.date_range(start_date, end_date, freq="D")
all_zips   = pd.Index(sorted(taxi_zip_max["zip_code"].dropna().unique()))
peak_levels = pd.Index([0, 1], name="is_peak")

full_index = pd.MultiIndex.from_product(
    [full_dates, all_zips, peak_levels],
    names=["date", "zip_code", "is_peak"]
)

taxi_zip_max = (
    taxi_zip_max
    .set_index(["date", "zip_code", "is_peak"])
    .reindex(full_index)
    .reset_index()
)

taxi_zip_max["traffic_count"] = taxi_zip_max["traffic_count"].fillna(0).astype("int32")

print("Method 1 completed.")
print(taxi_zip_max.shape)
display(taxi_zip_max.head())

Max-overlap crosswalk:


,taxi_zone_id,zip_code
2,2,99999
5,3,10469
8,4,10009
9,5,10312
11,6,10305


Method 1 completed.
(683904, 4)


,date,zip_code,is_peak,traffic_count
0,2020-01-01,10001,0,49482
1,2020-01-01,10001,1,11823
2,2020-01-01,10002,0,24742
3,2020-01-01,10002,1,3823
4,2020-01-01,10003,0,52536


### Method 2: Weighted

In [8]:
# --- 1. Build weighted crosswalk ---
weighted_map = zone_zip_intersection[
    ["taxi_zone_id", "zip_code", "zone_to_zip_weight"]
].copy()

# --- 2. Merge taxi counts to weighted ZIP mapping ---
taxi_zip_weighted = taxi_base.merge(
    weighted_map,
    on="taxi_zone_id",
    how="left"
)

taxi_zip_weighted = taxi_zip_weighted.dropna(subset=["zip_code", "zone_to_zip_weight"]).copy()
taxi_zip_weighted["zip_code"] = taxi_zip_weighted["zip_code"].astype(str)

# --- 3. Weighted traffic allocation ---
taxi_zip_weighted["weighted_traffic"] = (
    taxi_zip_weighted["traffic_count"] * taxi_zip_weighted["zone_to_zip_weight"]
)

# --- 4. Aggregate to ZIP × date × is_peak ---
taxi_zip_weighted = (
    taxi_zip_weighted
    .groupby(["date", "zip_code", "is_peak"], as_index=False)["weighted_traffic"]
    .sum()
)

# You may keep float, or round to integer for presentation
taxi_zip_weighted["traffic_count"] = taxi_zip_weighted["weighted_traffic"].round().astype("int32")
taxi_zip_weighted = taxi_zip_weighted.drop(columns=["weighted_traffic"])

# --- 5. Fill missing ZIP × date × is_peak with 0 ---
taxi_zip_weighted["date"] = pd.to_datetime(taxi_zip_weighted["date"], errors="coerce")

full_dates = pd.date_range(start_date, end_date, freq="D")
all_zips   = pd.Index(sorted(taxi_zip_weighted["zip_code"].dropna().unique()))
peak_levels = pd.Index([0, 1], name="is_peak")

full_index = pd.MultiIndex.from_product(
    [full_dates, all_zips, peak_levels],
    names=["date", "zip_code", "is_peak"]
)

taxi_zip_weighted = (
    taxi_zip_weighted
    .set_index(["date", "zip_code", "is_peak"])
    .reindex(full_index)
    .reset_index()
)

taxi_zip_weighted["traffic_count"] = taxi_zip_weighted["traffic_count"].fillna(0).astype("int32")

print("Method 2 completed.")
print(taxi_zip_weighted.shape)
display(taxi_zip_weighted.head())

Method 2 completed.
(780352, 4)


,date,zip_code,is_peak,traffic_count
0,2020-01-01,10001,0,41878
1,2020-01-01,10001,1,10308
2,2020-01-01,10002,0,26122
3,2020-01-01,10002,1,4114
4,2020-01-01,10003,0,33833


### Save boths methods output to shapefiles
You can upload shapefiles to ArcGIS for visualization.

In [53]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed_shp"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ZIP geometry back to display CRS
gdf_zip_display = gdf_zip.copy().to_crs(epsg=4326)

# --- Method 1 display table: average traffic by ZIP and peak flag ---
disp_max = (
    taxi_zip_max
    .groupby(["zip_code", "is_peak"], as_index=False)["traffic_count"]
    .mean()
)

disp_max = disp_max.rename(columns={"traffic_count": "avg_trfc"})
disp_max["avg_trfc"] = disp_max["avg_trfc"].round(2)

gdf_max = gdf_zip_display.merge(disp_max, on="zip_code", how="left")
gdf_max["avg_trfc"] = gdf_max["avg_trfc"].fillna(0)

# --- Method 2 display table ---
disp_weighted = (
    taxi_zip_weighted
    .groupby(["zip_code", "is_peak"], as_index=False)["traffic_count"]
    .mean()
)

disp_weighted = disp_weighted.rename(columns={"traffic_count": "avg_trfc"})
disp_weighted["avg_trfc"] = disp_weighted["avg_trfc"].round(2)

gdf_weighted = gdf_zip_display.merge(disp_weighted, on="zip_code", how="left")
gdf_weighted["avg_trfc"] = gdf_weighted["avg_trfc"].fillna(0)

# --- Save shapefiles ---
# Shapefile field names should be short
max_path = OUTPUT_DIR / "zip_max_overlap.shp"
wgt_path = OUTPUT_DIR / "zip_area_weight.shp"

gdf_max.to_file(max_path)
gdf_weighted.to_file(wgt_path)

print("Saved shapefiles:")
print(max_path)
print(wgt_path)

Saved shapefiles:
c:\Users\RyanW\OneDrive\Desktop\INFO 5368 PAML\Final Project\data\processed_shp\zip_max_overlap.shp
c:\Users\RyanW\OneDrive\Desktop\INFO 5368 PAML\Final Project\data\processed_shp\zip_area_weight.shp


## Proprocess Crash and Weather Data
### Load crash and weather data
Read raw data from .csv files (downloaded from NYC opendata).

In [9]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "original_data"

files = {
    "crash": DATA_DIR / "Motor_Vehicle_Collisions_-_Crashes_20260418.csv",
    "weather": DATA_DIR / "NYC Central Park Station 2020_2026.csv",
}

# Check files exist
for name, path in files.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} file not found: {path}")

# Load datasets
df_crash = pd.read_csv(files["crash"])
df_weather = pd.read_csv(files["weather"])

print("Loaded datasets successfully:")
for name, df in [("crash", df_crash), ("weather", df_weather)]:
    print(f"{name:8s} -> shape={df.shape}")

print("\nSample rows:")
display(df_crash.head(3))
display(df_weather.head(3))

Loaded datasets successfully:
crash    -> shape=(600478, 29)
weather  -> shape=(2288, 23)

Sample rows:


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,01/02/2020,0:00,NaN,NaN,NaN,NaN,NaN,CROSS ISLAND PARKWAY,NaN,NaN,...,NaN,NaN,NaN,NaN,4267700,Sedan,NaN,NaN,NaN,NaN
1,01/02/2020,12:57,NaN,NaN,NaN,NaN,NaN,W 57 & 8th Ave,W 57,NaN,...,Unspecified,NaN,NaN,NaN,4268255,Taxi,Pick-up Truck,NaN,NaN,NaN
2,01/02/2020,20:24,NaN,NaN,NaN,NaN,NaN,HUTCHINSON RIVER PARKWAY,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4268404,Station Wagon/Sport Utility Vehicle,Sedan,NaN,NaN,NaN


,STATION,NAME,DATE,AWND,PGTM,PRCP,SNOW,SNWD,TAVG,TMAX,...,WDF5,WSF2,WSF5,WT01,WT02,WT03,WT04,WT05,WT06,WT08
0,USW00094728,"NY CITY CENTRAL PARK, NY US",2020/1/1,8.50,NaN,0.00,0.0,0.0,NaN,41,...,260.0,17.0,29.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,USW00094728,"NY CITY CENTRAL PARK, NY US",2020/1/2,5.37,NaN,0.00,0.0,0.0,NaN,49,...,220.0,13.0,21.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,USW00094728,"NY CITY CENTRAL PARK, NY US",2020/1/3,3.36,NaN,0.15,0.0,0.0,NaN,49,...,230.0,10.1,15.0,1.0,NaN,NaN,NaN,NaN,NaN,1.0


## Data Quality Check
### TLC taxi data quality check
Check invalid data: date before 2020/01/01.

In [10]:
# --- 1. Copy and ensure datetime type ---
taxi_qc = taxi_base.copy()
taxi_qc["date"] = pd.to_datetime(taxi_qc["date"], errors="coerce")

# --- 2. Remove invalid records before 2020-01-01 ---
start_date = pd.Timestamp("2020-01-01")
end_date = pd.Timestamp("2025-12-31")

before_drop = len(taxi_qc)
taxi_qc = taxi_qc[taxi_qc["date"] >= start_date].copy()
after_drop = len(taxi_qc)

print("=== Taxi data temporal cleaning ===")
print(f"Rows before dropping pre-2020 data : {before_drop}")
print(f"Rows after dropping pre-2020 data  : {after_drop}")
print(f"Rows removed                       : {before_drop - after_drop}")

# --- 3. Keep only modeling window ---
taxi_qc = taxi_qc[taxi_qc["date"] <= end_date].copy()

# --- 4. Basic checks ---
total_rows = len(taxi_qc)
unique_dates = taxi_qc["date"].nunique()
unique_zones = taxi_qc["taxi_zone_id"].nunique()

print("\n=== Taxi base summary ===")
print(f"Rows in modeling window            : {total_rows}")
print(f"Unique dates                       : {unique_dates}")
print(f"Unique taxi zones                  : {unique_zones}")
print(f"Observed date range                : {taxi_qc['date'].min()} to {taxi_qc['date'].max()}")

# --- 5. Build full date × taxi_zone × is_peak grid ---
full_dates = pd.date_range(start_date, end_date, freq="D")
all_zones = pd.Index(sorted(taxi_qc["taxi_zone_id"].dropna().unique()))
peak_levels = pd.Index([0, 1], name="is_peak")

full_index = pd.MultiIndex.from_product(
    [full_dates, all_zones, peak_levels],
    names=["date", "taxi_zone_id", "is_peak"]
)

observed_index = pd.MultiIndex.from_frame(
    taxi_qc[["date", "taxi_zone_id", "is_peak"]].drop_duplicates()
)

missing_index = full_index.difference(observed_index)

missing_count = len(missing_index)
total_count = len(full_index)
missing_ratio = missing_count / total_count if total_count > 0 else float("nan")

print("\n=== Missingness in date × taxi_zone_id × is_peak panel ===")
print(f"Total possible combinations        : {total_count}")
print(f"Observed combinations              : {len(observed_index)}")
print(f"Missing combinations               : {missing_count}")
print(f"Missing ratio                      : {missing_ratio:.4%}")

# --- 6. Daily missingness summary ---
observed_per_day = (
    taxi_qc.groupby("date")[["taxi_zone_id", "is_peak"]]
    .apply(lambda x: len(x.drop_duplicates()))
    .reindex(full_dates, fill_value=0)
)

daily_missing = pd.DataFrame({
    "date": full_dates,
    "observed_zone_peak_count": observed_per_day.values
})

daily_missing["missing_zone_peak_count"] = len(all_zones) * 2 - daily_missing["observed_zone_peak_count"]
daily_missing["missing_ratio"] = daily_missing["missing_zone_peak_count"] / (len(all_zones) * 2)

print("\n=== Daily missingness summary ===")
print(f"Average missing zone-peak pairs per day : {daily_missing['missing_zone_peak_count'].mean():.2f}")
print(f"Average daily missing ratio             : {daily_missing['missing_ratio'].mean():.4%}")

print("\nWorst 10 days by missing ratio:")
display(daily_missing.sort_values("missing_ratio", ascending=False).head(10))

# Optional: replace original table
taxi_base = taxi_qc.sort_values(["date", "taxi_zone_id", "is_peak"]).reset_index(drop=True)

=== Taxi data temporal cleaning ===
Rows before dropping pre-2020 data : 1144298
Rows after dropping pre-2020 data  : 1143457
Rows removed                       : 841

=== Taxi base summary ===
Rows in modeling window            : 1143039
Unique dates                       : 2192
Unique taxi zones                  : 265
Observed date range                : 2020-01-01 00:00:00 to 2025-12-31 00:00:00

=== Missingness in date × taxi_zone_id × is_peak panel ===
Total possible combinations        : 1161760
Observed combinations              : 1143039
Missing combinations               : 18721
Missing ratio                      : 1.6114%

=== Daily missingness summary ===
Average missing zone-peak pairs per day : 8.54
Average daily missing ratio             : 1.6114%

Worst 10 days by missing ratio:


,date,observed_zone_peak_count,missing_zone_peak_count,missing_ratio
84,2020-03-25,517,13,0.024528
102,2020-04-12,517,13,0.024528
82,2020-03-23,517,13,0.024528
100,2020-04-10,517,13,0.024528
111,2020-04-21,517,13,0.024528
116,2020-04-26,517,13,0.024528
95,2020-04-05,517,13,0.024528
98,2020-04-08,518,12,0.022642
106,2020-04-16,518,12,0.022642
359,2020-12-25,518,12,0.022642


### Fill missing taxi data with count=0

In [11]:
# --- 1. Ensure datetime type ---
taxi_full = taxi_base.copy()
taxi_full["date"] = pd.to_datetime(taxi_full["date"])

# --- 2. Define full panel (date × taxi_zone_id × is_peak) ---
start_date = pd.Timestamp("2020-01-01")
end_date   = pd.Timestamp("2025-12-31")

full_dates = pd.date_range(start_date, end_date, freq="D")
all_zones = pd.Index(sorted(taxi_full["taxi_zone_id"].unique()))
peak_levels = pd.Index([0, 1], name="is_peak")

full_index = pd.MultiIndex.from_product(
    [full_dates, all_zones, peak_levels],
    names=["date", "taxi_zone_id", "is_peak"]
)

# --- 3. Reindex to full panel ---
taxi_full = (
    taxi_full
    .set_index(["date", "taxi_zone_id", "is_peak"])
    .reindex(full_index)
    .reset_index()
)

# --- 4. Fill missing traffic with 0 ---
missing_before = taxi_full["traffic_count"].isna().sum()
taxi_full["traffic_count"] = taxi_full["traffic_count"].fillna(0)
missing_after = taxi_full["traffic_count"].isna().sum()

# --- 5. Convert type ---
taxi_full["traffic_count"] = taxi_full["traffic_count"].astype("int32")
taxi_full["is_peak"] = taxi_full["is_peak"].astype("int8")

# --- 6. Report ---
print("=== Taxi data completion ===")
print(f"Total rows after completion        : {len(taxi_full)}")
print(f"Missing traffic before fill        : {missing_before}")
print(f"Missing traffic after fill         : {missing_after}")

print("\nSample rows:")
display(taxi_full.head())

# --- 7. Replace original table ---
taxi_base = taxi_full.sort_values(["date", "taxi_zone_id", "is_peak"]).reset_index(drop=True)

=== Taxi data completion ===
Total rows after completion        : 1161760
Missing traffic before fill        : 18721
Missing traffic after fill         : 0

Sample rows:


,date,taxi_zone_id,is_peak,traffic_count
0,2020-01-01,1,0,3333
1,2020-01-01,1,1,1651
2,2020-01-01,2,0,3
3,2020-01-01,2,1,0
4,2020-01-01,3,0,2233


### Weather data quality check

In [12]:
# --- 1. Copy raw weather data ---
weather_qc = df_weather.copy()

# --- 2. Columns we plan to use ---
required_cols = [
    "DATE",   # join key
    "AWND",   # average wind speed
    "PRCP",   # precipitation
    "SNOW",   # snowfall
    "SNWD",   # snow depth
    "TMAX",   # max temperature
    "TMIN"    # min temperature
]
# For WT -extreme weather, missing data is OK.

missing_required_cols = [c for c in required_cols if c not in weather_qc.columns]
if missing_required_cols:
    raise ValueError(f"Missing required weather columns: {missing_required_cols}")

# --- 3. Parse DATE ---
weather_qc["DATE"] = pd.to_datetime(weather_qc["DATE"], errors="coerce")

# --- 4. Convert required numeric columns ---
numeric_cols = ["AWND", "PRCP", "SNOW", "SNWD", "TMAX", "TMIN"]
for col in numeric_cols:
    weather_qc[col] = pd.to_numeric(weather_qc[col], errors="coerce")

# --- 5. Basic table-level checks ---
total_rows = len(weather_qc)
date_na_count = weather_qc["DATE"].isna().sum()

date_min = weather_qc["DATE"].min()
date_max = weather_qc["DATE"].max()

dup_date_count = weather_qc.duplicated(subset=["DATE"]).sum()

# Restrict the completeness check to the modeling window
model_start = pd.Timestamp("2020-01-01")
model_end   = pd.Timestamp("2025-12-31")

weather_model = weather_qc[
    (weather_qc["DATE"] >= model_start) &
    (weather_qc["DATE"] <= model_end)
].copy()

full_dates = pd.date_range(model_start, model_end, freq="D")
observed_dates = pd.DatetimeIndex(weather_model["DATE"].dropna().unique()).sort_values()
missing_dates = full_dates.difference(observed_dates)

# --- 6. Missing-value summary for required columns ---
missing_summary = pd.DataFrame({
    "column": required_cols,
    "missing_count": [weather_model[c].isna().sum() for c in required_cols],
})

missing_summary["missing_ratio"] = missing_summary["missing_count"] / len(weather_model)

# --- 7. Optional sanity checks for impossible-looking values ---
range_check = pd.DataFrame({
    "column": numeric_cols,
    "min_value": [weather_model[c].min() for c in numeric_cols],
    "max_value": [weather_model[c].max() for c in numeric_cols],
    "mean_value": [weather_model[c].mean() for c in numeric_cols],
})

# --- 8. Report ---
print("=== Weather Data Quality Check ===")
print(f"Total rows in raw table             : {total_rows}")
print(f"DATE parse failures                 : {date_na_count}")
print(f"Raw date range                      : {date_min} to {date_max}")
print(f"Duplicate DATE rows (raw)           : {dup_date_count}")

print("\n=== Modeling window coverage (2020-01-01 to 2025-12-31) ===")
print(f"Rows in modeling window             : {len(weather_model)}")
print(f"Expected number of calendar days    : {len(full_dates)}")
print(f"Observed unique dates               : {len(observed_dates)}")
print(f"Missing dates in window             : {len(missing_dates)}")
print(f"Missing date ratio                  : {len(missing_dates) / len(full_dates):.4%}")

if len(missing_dates) > 0:
    print("\nSample missing dates:")
    print(missing_dates[:10])

print("\n=== Missingness in required columns (WT* excluded) ===")
display(missing_summary.sort_values("missing_ratio", ascending=False))

print("\n=== Numeric range summary ===")
display(range_check)

# Optional: show rows with any missing value in required columns
rows_with_missing_required = weather_model[required_cols].isna().any(axis=1).sum()
print(f"\nRows with any missing required field: {rows_with_missing_required}")

=== Weather Data Quality Check ===
Total rows in raw table             : 2288
DATE parse failures                 : 0
Raw date range                      : 2020-01-01 00:00:00 to 2026-04-06 00:00:00
Duplicate DATE rows (raw)           : 0

=== Modeling window coverage (2020-01-01 to 2025-12-31) ===
Rows in modeling window             : 2192
Expected number of calendar days    : 2192
Observed unique dates               : 2192
Missing dates in window             : 0
Missing date ratio                  : 0.0000%

=== Missingness in required columns (WT* excluded) ===


,column,missing_count,missing_ratio
1,AWND,59,0.026916
4,SNWD,3,0.001369
0,DATE,0,0.000000
2,PRCP,0,0.000000
3,SNOW,0,0.000000
5,TMAX,0,0.000000
6,TMIN,0,0.000000



=== Numeric range summary ===


,column,min_value,max_value,mean_value
0,AWND,0.67,14.32,5.068556
1,PRCP,0.00,7.13,0.135315
2,SNOW,0.00,14.80,0.040465
3,SNWD,0.00,14.20,0.148470
4,TMAX,15.00,99.00,64.080748
5,TMIN,3.00,81.00,50.059307



Rows with any missing required field: 62


### Fill missing weather data
For AWND, we use linear interpolation to fill missing data.
For SNWD, we replace missing data with 0.

In [13]:
# --- 1. Copy and prepare weather data ---
weather_clean = df_weather.copy()

# Ensure DATE is datetime
weather_clean["DATE"] = pd.to_datetime(weather_clean["DATE"], errors="coerce")

# Keep only rows with valid DATE and sort by date
weather_clean = weather_clean.dropna(subset=["DATE"]).sort_values("DATE").reset_index(drop=True)

# Ensure numeric dtype
weather_clean["AWND"] = pd.to_numeric(weather_clean["AWND"], errors="coerce")
weather_clean["SNWD"] = pd.to_numeric(weather_clean["SNWD"], errors="coerce")

# --- 2. Record missing counts before imputation ---
awnd_missing_before = weather_clean["AWND"].isna().sum()
snwd_missing_before = weather_clean["SNWD"].isna().sum()

# --- 3. Impute AWND with linear interpolation ---
# This uses previous and next valid observations, and works for consecutive missing days.
weather_clean["AWND"] = weather_clean["AWND"].interpolate(method="linear", limit_direction="both")

# --- 4. Impute SNWD with 0 ---
weather_clean["SNWD"] = weather_clean["SNWD"].fillna(0)

# --- 5. Record missing counts after imputation ---
awnd_missing_after = weather_clean["AWND"].isna().sum()
snwd_missing_after = weather_clean["SNWD"].isna().sum()

# --- 6. Report ---
print("=== Weather missing-value imputation completed ===")
print(f"AWND missing before: {awnd_missing_before}")
print(f"AWND missing after : {awnd_missing_after}")
print(f"SNWD missing before: {snwd_missing_before}")
print(f"SNWD missing after : {snwd_missing_after}")

# Optional: preview rows that originally had missing AWND or SNWD
print("\nSample rows after imputation:")
display(weather_clean.loc[:, ["DATE", "AWND", "SNWD"]].head(10))

# Optional: replace original table for downstream use
df_weather = weather_clean

=== Weather missing-value imputation completed ===
AWND missing before: 60
AWND missing after : 0
SNWD missing before: 3
SNWD missing after : 0

Sample rows after imputation:


,DATE,AWND,SNWD
0,2020-01-01,8.50,0.0
1,2020-01-02,5.37,0.0
2,2020-01-03,3.36,0.0
3,2020-01-04,4.47,0.0
4,2020-01-05,11.41,0.0
5,2020-01-06,6.04,0.0
6,2020-01-07,5.14,0.0
7,2020-01-08,9.62,0.0
8,2020-01-09,4.47,0.0
9,2020-01-10,6.26,0.0


In [14]:
weather_temp = df_weather.copy()

# --- Ensure numeric ---
weather_temp["TMAX"] = pd.to_numeric(weather_temp["TMAX"], errors="coerce")
weather_temp["TMIN"] = pd.to_numeric(weather_temp["TMIN"], errors="coerce")

# --- Compute average temperature ---
weather_temp["TAVG"] = (weather_temp["TMAX"] + weather_temp["TMIN"]) / 2

# --- Drop original columns ---
weather_temp = weather_temp.drop(columns=["TMAX", "TMIN"])

# --- Report ---
print("=== Temperature feature updated ===")
print(f"Missing TAVG count: {weather_temp['TAVG'].isna().sum()}")

print("\nSample:")
display(weather_temp[["DATE", "TAVG"]].head())

# --- Replace original dataframe ---
df_weather = weather_temp

=== Temperature feature updated ===
Missing TAVG count: 0

Sample:


,DATE,TAVG
0,2020-01-01,37.5
1,2020-01-02,41.0
2,2020-01-03,46.5
3,2020-01-04,46.0
4,2020-01-05,38.5


### Crash data geocoding

In [15]:
# --- 1. Copy raw crash data ---
crash_work = df_crash.copy()

# Keep original row count for reporting
n_raw = len(crash_work)

# --- 2. Parse coordinates ---
crash_work["LATITUDE"] = pd.to_numeric(crash_work["LATITUDE"], errors="coerce")
crash_work["LONGITUDE"] = pd.to_numeric(crash_work["LONGITUDE"], errors="coerce")

# Drop missing or zero coordinates
valid_coord_mask = (
    crash_work["LATITUDE"].notna() &
    crash_work["LONGITUDE"].notna() &
    (crash_work["LATITUDE"] != 0) &
    (crash_work["LONGITUDE"] != 0)
)

crash_work = crash_work[valid_coord_mask].copy()
n_after_coord = len(crash_work)

# --- 3. Parse date and time ---
crash_work["CRASH DATE"] = pd.to_datetime(crash_work["CRASH DATE"], errors="coerce")

# Parse CRASH TIME safely
crash_work["CRASH TIME"] = pd.to_datetime(
    crash_work["CRASH TIME"],
    format="%H:%M",
    errors="coerce"
)

# Drop rows with invalid date/time
crash_work = crash_work.dropna(subset=["CRASH DATE", "CRASH TIME"]).copy()
n_after_datetime = len(crash_work)

# --- 4. Derive date / hour / minute / is_peak ---
crash_work["date"] = crash_work["CRASH DATE"].dt.normalize()
crash_work["hour"] = crash_work["CRASH TIME"].dt.hour
crash_work["minute"] = crash_work["CRASH TIME"].dt.minute

# Peak definition:
# morning peak: 07:00–09:30
# evening peak: 16:00–19:00
crash_work["is_peak"] = (
    ((crash_work["hour"] == 7) | (crash_work["hour"] == 8) | ((crash_work["hour"] == 9) & (crash_work["minute"] <= 30))) |
    ((crash_work["hour"] == 16) | (crash_work["hour"] == 17) | (crash_work["hour"] == 18) | ((crash_work["hour"] == 19) & (crash_work["minute"] == 0)))
).astype("int8")

# --- 5. Convert crash table to GeoDataFrame ---
gdf_crash = gpd.GeoDataFrame(
    crash_work,
    geometry=gpd.points_from_xy(crash_work["LONGITUDE"], crash_work["LATITUDE"]),
    crs="EPSG:4326"
)

# Ensure ZIP polygons are also in EPSG:4326 for point-in-polygon join
gdf_zip_join = gdf_zip.to_crs(epsg=4326).copy()

# --- 6. Spatial join: crash point -> ZIP ---
gdf_crash_zip = gpd.sjoin(
    gdf_crash,
    gdf_zip_join[["zip_code", "geometry"]],
    how="left",
    predicate="within"
)

# Drop crash records that could not be assigned to a ZIP polygon
gdf_crash_zip = gdf_crash_zip.dropna(subset=["zip_code"]).copy()
gdf_crash_zip["zip_code"] = gdf_crash_zip["zip_code"].astype(str).str.strip()

n_after_zip = len(gdf_crash_zip)

# --- 7. Keep essential columns only (not yet aggregated) ---
# Keep COLLISION_ID if available for later dedup / traceability
keep_cols = ["date", "is_peak", "zip_code", "LATITUDE", "LONGITUDE", "geometry"]
if "COLLISION_ID" in gdf_crash_zip.columns:
    keep_cols = ["COLLISION_ID"] + keep_cols

crash_assigned = gdf_crash_zip[keep_cols].copy()

# --- 8. Report dropping statistics ---
n_dropped_total = n_raw - n_after_zip
drop_ratio_total = n_dropped_total / n_raw if n_raw > 0 else float("nan")

print("=== Crash spatial assignment completed ===")
print(f"Raw crash rows                              : {n_raw}")
print(f"Rows after removing missing/zero coords     : {n_after_coord}")
print(f"Rows after removing invalid date/time        : {n_after_datetime}")
print(f"Rows successfully assigned to ZIP            : {n_after_zip}")
print(f"Total rows dropped                           : {n_dropped_total}")
print(f"Total dropped ratio                          : {drop_ratio_total:.4%}")

print("\nAssigned crash sample:")
display(crash_assigned.head())

# Optional: replace / store for next cell
# crash_assigned is the non-aggregated crash table with zip_code + is_peak

=== Crash spatial assignment completed ===
Raw crash rows                              : 600478
Rows after removing missing/zero coords     : 552898
Rows after removing invalid date/time        : 552898
Rows successfully assigned to ZIP            : 551405
Total rows dropped                           : 49073
Total dropped ratio                          : 8.1723%

Assigned crash sample:


,COLLISION_ID,date,is_peak,zip_code,LATITUDE,LONGITUDE,geometry
4,4268117,2020-01-02,1,11201,40.699955,-73.98682,POINT (-73.98682 40.69996)
5,4268477,2020-01-02,0,11373,40.739944,-73.87530,POINT (-73.8753 40.73994)
6,4268282,2020-01-02,1,11234,40.632908,-73.92736,POINT (-73.92736 40.63291)
7,4267865,2020-01-02,1,11358,40.757732,-73.79404,POINT (-73.79404 40.75773)
8,4270000,2020-01-02,0,11233,40.678726,-73.91906,POINT (-73.91906 40.67873)


## Align Data
Align taxi/Crash/weather data together, based on weighted taxi data.

In [16]:
# --------------------------------------------------
# 0. Choose which taxi table to use
#    Options: "max" or "weighted"
# --------------------------------------------------
TRAFFIC_SOURCE = "weighted"

if TRAFFIC_SOURCE == "max":
    traffic_df = taxi_zip_max.copy()
elif TRAFFIC_SOURCE == "weighted":
    traffic_df = taxi_zip_weighted.copy()
else:
    raise ValueError("TRAFFIC_SOURCE must be either 'max' or 'weighted'")

# --------------------------------------------------
# 1. Prepare traffic table
# --------------------------------------------------
traffic_df["date"] = pd.to_datetime(traffic_df["date"], errors="coerce")
traffic_df["zip_code"] = traffic_df["zip_code"].astype(str).str.strip()
traffic_df["is_peak"] = pd.to_numeric(traffic_df["is_peak"], errors="coerce").fillna(0).astype("int8")
traffic_df["traffic_count"] = pd.to_numeric(traffic_df["traffic_count"], errors="coerce").fillna(0)

# --------------------------------------------------
# 2. Aggregate crash data to zip_code × date × is_peak
# --------------------------------------------------
crash_grouped = crash_assigned.copy()

crash_grouped["date"] = pd.to_datetime(crash_grouped["date"], errors="coerce")
crash_grouped["zip_code"] = crash_grouped["zip_code"].astype(str).str.strip()
crash_grouped["is_peak"] = pd.to_numeric(crash_grouped["is_peak"], errors="coerce").fillna(0).astype("int8")

if "COLLISION_ID" in crash_grouped.columns:
    crash_grouped = (
        crash_grouped
        .groupby(["date", "zip_code", "is_peak"], as_index=False)["COLLISION_ID"]
        .nunique()
        .rename(columns={"COLLISION_ID": "crash_count"})
    )
else:
    crash_grouped = (
        crash_grouped
        .groupby(["date", "zip_code", "is_peak"], as_index=False)
        .size()
        .rename(columns={"size": "crash_count"})
    )

# --------------------------------------------------
# 3. Prepare weather table
# --------------------------------------------------
weather_merge = df_weather.copy()
weather_merge["DATE"] = pd.to_datetime(weather_merge["DATE"], errors="coerce")

# Base weather columns
base_weather_cols = ["DATE", "TAVG", "AWND", "PRCP", "SNOW", "SNWD"]

# Identify all WT extreme-weather columns automatically
wt_cols = [c for c in weather_merge.columns if str(c).startswith("WT")]
wt_cols = sorted(wt_cols)

# One-hot / indicator cleanup for WT columns
for col in wt_cols:
    weather_merge[col] = pd.to_numeric(weather_merge[col], errors="coerce").fillna(0)
    weather_merge[col] = (weather_merge[col] > 0).astype("int8")

# Keep weather + WT one-hot columns
weather_cols = [c for c in base_weather_cols if c in weather_merge.columns] + wt_cols
weather_merge = weather_merge[weather_cols].copy()

# --------------------------------------------------
# 4. Merge traffic + crash
# --------------------------------------------------
final_df = traffic_df.merge(
    crash_grouped,
    on=["date", "zip_code", "is_peak"],
    how="left"
)

final_df["crash_count"] = final_df["crash_count"].fillna(0).astype("int32")

# --------------------------------------------------
# 5. Merge weather by date
# --------------------------------------------------
final_df = final_df.merge(
    weather_merge,
    left_on="date",
    right_on="DATE",
    how="left"
)

if "DATE" in final_df.columns:
    final_df = final_df.drop(columns=["DATE"])

# --------------------------------------------------
# 6. Fill missing values after weather merge
# --------------------------------------------------
numeric_weather_cols = ["TAVG", "AWND", "PRCP", "SNOW", "SNWD"]
for col in numeric_weather_cols:
    if col in final_df.columns:
        final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

for col in wt_cols:
    if col in final_df.columns:
        final_df[col] = final_df[col].fillna(0).astype("int8")

# --------------------------------------------------
# 7. Final sort and report
# --------------------------------------------------
final_df = final_df.sort_values(["date", "zip_code", "is_peak"]).reset_index(drop=True)

print("=== Final merged modeling table ===")
print(f"Traffic source used : {TRAFFIC_SOURCE}")
print(f"Number of rows      : {len(final_df)}")
print(f"Number of columns   : {final_df.shape[1]}")
print(f"WT columns found    : {wt_cols}")

if wt_cols:
    print("\nWT column sums:")
    display(final_df[wt_cols].sum().to_frame("count"))
else:
    print("\nNo WT columns found in weather data.")

print("\nSample rows:")
display(final_df.head())

=== Final merged modeling table ===
Traffic source used : weighted
Number of rows      : 780352
Number of columns   : 17
WT columns found    : ['WT01', 'WT02', 'WT03', 'WT04', 'WT05', 'WT06', 'WT08']

WT column sums:


,count
WT01,275544
WT02,24564
WT03,55892
WT04,2136
WT05,356
WT06,3204
WT08,123532



Sample rows:


,date,zip_code,is_peak,traffic_count,crash_count,TAVG,AWND,PRCP,SNOW,SNWD,WT01,WT02,WT03,WT04,WT05,WT06,WT08
0,2020-01-01,10001,0,41878,0,37.5,8.5,0.0,0.0,0.0,0,0,0,0,0,0,0
1,2020-01-01,10001,1,10308,0,37.5,8.5,0.0,0.0,0.0,0,0,0,0,0,0,0
2,2020-01-01,10002,0,26122,0,37.5,8.5,0.0,0.0,0.0,0,0,0,0,0,0,0
3,2020-01-01,10002,1,4114,0,37.5,8.5,0.0,0.0,0.0,0,0,0,0,0,0,0
4,2020-01-01,10003,0,33833,0,37.5,8.5,0.0,0.0,0.0,0,0,0,0,0,0,0


### Transfer data to weekday

In [17]:
df_model = final_df.copy()

# --- Ensure datetime ---
df_model["date"] = pd.to_datetime(df_model["date"], errors="coerce")

# --- Weekday encoding ---
df_model["weekday"] = df_model["date"].dt.weekday  # 0=Mon, ..., 6=Sun

# --- Report ---
print("=== Weekday feature created ===")
print("Unique weekday values:", sorted(df_model["weekday"].dropna().unique()))

print("\nSample:")
display(df_model[["date", "weekday"]].head())

# --- Drop original date column ---
df_model = df_model.drop(columns=["date"])

# --- Replace original table ---
final_df = df_model

=== Weekday feature created ===
Unique weekday values: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]

Sample:


,date,weekday
0,2020-01-01,2
1,2020-01-01,2
2,2020-01-01,2
3,2020-01-01,2
4,2020-01-01,2


### Save data to .csv

In [18]:
PROJECT_ROOT = Path.cwd()

OUTPUT_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- File path ---
output_path = OUTPUT_DIR / "data.csv"

# --- Save CSV ---
final_df.to_csv(output_path, index=False)

# --- Report ---
file_size_mb = output_path.stat().st_size / (1024 * 1024)

print("=== Dataset saved ===")
print(f"Path        : {output_path}")
print(f"File size   : {file_size_mb:.2f} MB")
print(f"Rows        : {len(final_df)}")
print(f"Columns     : {final_df.shape[1]}")

=== Dataset saved ===
Path        : c:\Users\RyanW\OneDrive\Desktop\INFO 5368 PAML\Final Project\data\data.csv
File size   : 40.45 MB
Rows        : 780352
Columns     : 17


## Data Engineering

The following 3 cells aim at data engineering.

本步骤对原始建模数据进行了特征工程处理，主要包括以下几类：

1. **计数变量变换**
   - 对 `traffic_count` 和 `crash_count` 使用 `log1p(x)` 变换，生成 `log_traffic_count` 和 `log_crash_count`。
   - 这样可以缓解原始计数分布的强右偏问题，降低极端大值对模型训练的影响。

2. **零膨胀天气变量处理**
   - 对降雨、降雪和积雪深度这类大量为 0 的变量进行了“是否发生 + 强度大小”的拆分：
     - `PRCP` → `is_rain` 和 `log_prcp`
     - `SNOW` → `is_snow` 和 `log_snow`
     - `SNWD` → `has_snow_depth` 和 `log_snwd`
   - 这样可以让模型同时学习“是否发生天气事件”和“天气强度大小”的不同影响。

3. **连续变量标准化**
   - 对连续天气变量 `TAVG` 和 `AWND` 进行了 z-score 标准化，生成 `tavg_z` 和 `awnd_z`。
   - 这样可以消除量纲差异，使不同连续特征处于相近尺度，提升模型训练稳定性。

4. **保留时序与场景特征**
   - 保留 `is_peak` 和 `weekday`，用于描述高峰时段与周内周期性变化。

经过上述处理后，数据既保留了原始字段，也新增了更适合建模的工程化特征，可直接用于后续回归或分类任务。

In this step, feature engineering was applied to the merged modeling table. The main transformations are:

1. **Count-variable transformation**
   - `traffic_count` and `crash_count` were transformed using `log1p(x)`, producing `log_traffic_count` and `log_crash_count`.
   - This helps reduce strong right-skewness and limits the influence of extreme large values during model training.

2. **Zero-inflated weather feature handling**
   - For precipitation, snowfall, and snow depth, which contain many zeros, each variable was decomposed into:
     - an occurrence indicator, and
     - a magnitude feature after log transformation.
   - Specifically:
     - `PRCP` → `is_rain` and `log_prcp`
     - `SNOW` → `is_snow` and `log_snow`
     - `SNWD` → `has_snow_depth` and `log_snwd`
   - This allows the model to separately learn whether a weather event occurred and how intense it was.

3. **Standardization of continuous weather variables**
   - `TAVG` and `AWND` were standardized using z-score normalization, producing `tavg_z` and `awnd_z`.
   - This reduces scale differences across continuous features and improves training stability.

4. **Preserving temporal/context features**
   - `is_peak` and `weekday` were kept as important temporal predictors that capture rush-hour effects and weekly patterns.

After these transformations, the dataset retains the original variables while adding model-ready engineered features for downstream regression or classification tasks.

### Load preprocessed data

In [19]:
# --- Path ---
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "data.csv"

# --- Load ---
df = pd.read_csv(DATA_PATH)

print("=== Data loaded ===")
print(f"Shape: {df.shape}")

# --- Exclude specific columns ---
exclude_cols = ["is_peak", "weekday"]
cols_to_check = [c for c in df.columns if c not in exclude_cols]

# --- Build summary ---
summary = []

for col in cols_to_check:
    col_data = df[col]
    
    col_type = col_data.dtype
    
    # Only compute min/max for numeric columns
    if pd.api.types.is_numeric_dtype(col_data):
        col_min = col_data.min()
        col_max = col_data.max()
    else:
        col_min = None
        col_max = None
    
    summary.append({
        "column": col,
        "dtype": str(col_type),
        "min": col_min,
        "max": col_max
    })

summary_df = pd.DataFrame(summary)

# --- Report ---
print("\n=== Column Summary ===")
display(summary_df)

=== Data loaded ===
Shape: (780352, 17)

=== Column Summary ===


,column,dtype,min,max
0,zip_code,int64,10001.00,99999.00
1,traffic_count,int64,0.00,60524.00
2,crash_count,int64,0.00,13.00
3,TAVG,float64,11.00,90.00
4,AWND,float64,0.67,14.32
5,PRCP,float64,0.00,7.13
6,SNOW,float64,0.00,14.80
7,SNWD,float64,0.00,14.20
8,WT01,int64,0.00,1.00
9,WT02,int64,0.00,1.00


### Feature engineering

In [21]:
# --- 1. Load data ---
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "data.csv"

df_fe = pd.read_csv(DATA_PATH)

print("=== Raw data loaded ===")
print(f"Shape before engineering: {df_fe.shape}")

# --- 2. Basic type cleanup ---
numeric_cols = [
    "traffic_count", "crash_count",
    "TAVG", "AWND", "PRCP", "SNOW", "SNWD",
    "is_peak", "weekday"
]

for col in numeric_cols:
    if col in df_fe.columns:
        df_fe[col] = pd.to_numeric(df_fe[col], errors="coerce")

if "zip_code" in df_fe.columns:
    df_fe["zip_code"] = df_fe["zip_code"].astype(str).str.strip()

# Keep WT one-hot extreme-weather columns
wt_cols = [c for c in df_fe.columns if str(c).startswith("WT")]
for col in wt_cols:
    df_fe[col] = pd.to_numeric(df_fe[col], errors="coerce").fillna(0).astype("int8")

# --- 3. Feature engineering ---
# Count features
df_fe["log_traffic_count"] = np.log1p(df_fe["traffic_count"])
df_fe["log_crash_count"] = np.log1p(df_fe["crash_count"])

# Weather: occurrence + intensity
df_fe["is_rain"] = (df_fe["PRCP"] > 0).astype("int8")
df_fe["log_prcp"] = np.log1p(df_fe["PRCP"])

df_fe["is_snow"] = (df_fe["SNOW"] > 0).astype("int8")
df_fe["log_snow"] = np.log1p(df_fe["SNOW"])

df_fe["has_snow_depth"] = (df_fe["SNWD"] > 0).astype("int8")
df_fe["log_snwd"] = np.log1p(df_fe["SNWD"])

# Standardization
for col in ["TAVG", "AWND"]:
    mean_val = df_fe[col].mean()
    std_val = df_fe[col].std()
    if pd.isna(std_val) or std_val == 0:
        df_fe[f"{col.lower()}_z"] = 0.0
    else:
        df_fe[f"{col.lower()}_z"] = (df_fe[col] - mean_val) / std_val

# --- 4. Drop original columns after transformation ---
cols_to_drop = [
    "traffic_count", "crash_count",
    "PRCP", "SNOW", "SNWD",
    "TAVG", "AWND"
]

df_fe = df_fe.drop(columns=[c for c in cols_to_drop if c in df_fe.columns])

# --- 5. Reorder columns (optional) ---
preferred_order = [
    "zip_code", "date", "weekday", "is_peak",
    "log_traffic_count", "log_crash_count",
    "is_rain", "log_prcp",
    "is_snow", "log_snow",
    "has_snow_depth", "log_snwd",
    "tavg_z", "awnd_z"
]

existing_cols = [c for c in preferred_order if c in df_fe.columns]
remaining_cols = [c for c in df_fe.columns if c not in existing_cols]
df_fe = df_fe[existing_cols + remaining_cols]

# --- 6. Report ---
print("\n=== Feature engineering completed ===")
print(f"Shape after engineering: {df_fe.shape}")

print("\nColumns after engineering:")
print(df_fe.columns.tolist())

print("\nMissing values summary:")
display(df_fe.isna().sum().to_frame("missing_count"))

print("\nSample rows:")
display(df_fe.head())

=== Raw data loaded ===
Shape before engineering: (780352, 17)

=== Feature engineering completed ===
Shape after engineering: (780352, 20)

Columns after engineering:
['zip_code', 'weekday', 'is_peak', 'log_traffic_count', 'log_crash_count', 'is_rain', 'log_prcp', 'is_snow', 'log_snow', 'has_snow_depth', 'log_snwd', 'tavg_z', 'awnd_z', 'WT01', 'WT02', 'WT03', 'WT04', 'WT05', 'WT06', 'WT08']

Missing values summary:


,missing_count
zip_code,0
weekday,0
is_peak,0
log_traffic_count,0
log_crash_count,0
is_rain,0
log_prcp,0
is_snow,0
log_snow,0
has_snow_depth,0



Sample rows:


,zip_code,weekday,is_peak,log_traffic_count,log_crash_count,is_rain,log_prcp,is_snow,log_snow,has_snow_depth,log_snwd,tavg_z,awnd_z,WT01,WT02,WT03,WT04,WT05,WT06,WT08
0,10001,2,0,10.642540,0.0,0,0.0,0,0.0,0,0.0,-1.199237,1.472595,0,0,0,0,0,0,0
1,10001,2,1,9.240773,0.0,0,0.0,0,0.0,0,0.0,-1.199237,1.472595,0,0,0,0,0,0,0
2,10002,2,0,10.170571,0.0,0,0.0,0,0.0,0,0.0,-1.199237,1.472595,0,0,0,0,0,0,0
3,10002,2,1,8.322394,0.0,0,0.0,0,0.0,0,0.0,-1.199237,1.472595,0,0,0,0,0,0,0
4,10003,2,0,10.429221,0.0,0,0.0,0,0.0,0,0.0,-1.199237,1.472595,0,0,0,0,0,0,0


### Save data to .csv

In [22]:
# --- Save engineered data ---
PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "data_engineering.csv"

df_fe.to_csv(OUTPUT_PATH, index=False)

print("=== Engineered dataset saved ===")
print(f"Path    : {OUTPUT_PATH}")
print(f"Rows    : {df_fe.shape[0]}")
print(f"Columns : {df_fe.shape[1]}")

=== Engineered dataset saved ===
Path    : c:\Users\RyanW\OneDrive\Desktop\INFO 5368 PAML\Final Project\data\data_engineering.csv
Rows    : 780352
Columns : 20
